# DataFileLoader — Companies House Extraction

Downloads the latest Companies House bulk snapshot, filters for hospitality SIC codes, deduplicates by company name, flags the latest incorporation per trading name, and merges FSA hygiene ratings.

**Output:** `Data/Final Dataset.xlsx`

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import requests
import io
import os
import re
from pathlib import Path

# ── Resolve repo root so all paths work whether run locally or in Colab ──
if '__file__' in dir():
    REPO_ROOT = Path(__file__).resolve().parent.parent
else:
    REPO_ROOT = Path().resolve()
os.chdir(REPO_ROOT)
DATA_DIR = REPO_ROOT / "Data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")


## 1. Download Companies House Bulk Snapshot

In [ ]:
ZIP_URL = "https://download.companieshouse.gov.uk/BasicCompanyDataAsOneFile-2026-02-01.zip"

print("Downloading Companies House snapshot...")
response = requests.get(ZIP_URL)
if response.status_code != 200:
    raise RuntimeError(f"Download failed — HTTP {response.status_code}")
print(f"  Downloaded {len(response.content)/1e6:.1f} MB")

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    csv_filename = z.namelist()[0]
    print(f"  Extracting: {csv_filename}")
    with z.open(csv_filename) as f:
        chunks = []
        for chunk in pd.read_csv(f, chunksize=100_000, low_memory=False):
            chunks.append(chunk)
        df = pd.concat(chunks, ignore_index=True)

print(f"  Loaded {len(df):,} companies, {df.shape[1]} columns")


## 2. Filter for Hospitality SIC Codes

In [ ]:
# SIC codes covering restaurants, cafes, pubs, fast food, catering
HOSPITALITY_SICS = ['56101','56102','56103','56210','56290','56301','56302']

def extract_sic(x):
    if not isinstance(x, str): return None
    m = re.search(r'\d+', x)
    return m.group(0) if m else None

df['SIC_Numeric'] = df['SICCode.SicText_1'].apply(extract_sic)
hospitality_df = df[df['SIC_Numeric'].isin(HOSPITALITY_SICS)].copy()

print(f"Hospitality businesses found : {len(hospitality_df):,}")
print("\nBreakdown by SIC code:")
print(hospitality_df['SICCode.SicText_1'].value_counts().to_string())


## 3. Status Summary — Active vs Dissolved

In [ ]:
status_counts = hospitality_df['CompanyStatus'].value_counts()
print("Company status breakdown:")
print(status_counts.to_string())
print(f"\nLiquidation rate (Liquidation / Active+Liquidation): "
      f"{status_counts.get('Liquidation',0) / (status_counts.get('Active',0)+status_counts.get('Liquidation',0)):.2%}")


## 4. Deduplicate — Keep Latest Incorporation Per Trading Name

In [ ]:
# Group by CompanyName, keep only the most-recently-incorporated entry per name
hospitality_df['IncorporationDate'] = pd.to_datetime(
    hospitality_df['IncorporationDate'], errors='coerce', dayfirst=True
)

list_of_dfs = [g for _, g in hospitality_df.groupby('CompanyName') if len(g) > 1]
patch_data = {}
for sub_df in list_of_dfs:
    if sub_df.empty: continue
    sub_df = sub_df.sort_values('IncorporationDate', ascending=False)
    for i, idx in enumerate(sub_df.index):
        patch_data[idx] = (i == 0)

patch_series = pd.Series(patch_data, name='Is_Latest_Match')
hospitality_df = hospitality_df.copy()
hospitality_df.update(patch_series.to_frame())
hospitality_df = hospitality_df[hospitality_df.get('Is_Latest_Match', True) != False]

print(f"After deduplication: {len(hospitality_df):,} businesses")
print(hospitality_df['CompanyStatus'].value_counts().to_string())


## 5. Merge FSA Hygiene Ratings

In [ ]:
# ratings_df must be available from a prior notebook or loaded here
ratings_path = DATA_DIR / "fsa_ratings.xlsx"   # adjust filename as needed
if ratings_path.exists():
    ratings_df = pd.read_excel(ratings_path)
    hospitality_df = hospitality_df.merge(
        ratings_df,
        how='left',
        left_on='Final_Business_ID',
        right_on='FHRSID'
    )
    print(f"Merged with hygiene ratings. Shape: {hospitality_df.shape}")
else:
    print("FSA ratings file not found — skipping merge. "
          "Run rating_gathering.ipynb first.")


## 6. Summary & Save

In [ ]:
print("=== Final Dataset Summary ===")
print(f"Rows    : {len(hospitality_df):,}")
print(f"Columns : {hospitality_df.shape[1]}")
print(f"\nNull counts (top 10):")
print(hospitality_df.isna().sum().sort_values(ascending=False).head(10).to_string())
print(f"\nStatus breakdown:")
print(hospitality_df['CompanyStatus'].value_counts().to_string())

out_path = DATA_DIR / "Final Dataset.xlsx"
hospitality_df.to_excel(out_path, index=False)
print(f"\n✓ Saved to {out_path}")
